# Explainability (XAI) for Perception: Methods, Sanity Checks and Limits as Safety Evidence

**Module 06 · AI Safety (ISO/PAS 8800)** · Notebook 6 of 8

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/Autonomous-Driving_AI-Safety-and-Security/blob/main/06_AI_Safety_ISO_PAS_8800/06_explainability.ipynb)

| Level | Prerequisites | Standards |
|---|---|---|
| Intermediate | [03 MC dropout and ensembles](03_mc_dropout_and_deep_ensembles.ipynb); CNN basics | ISO/PAS 8800:2024, ISO/IEC TR 5469:2024 |

**Author:** Milin Patel · Hochschule Kempten · MIT Licence

## Learning objectives

- Understand where explainability supports safety work: debugging, safety analysis of AI errors, incident investigation and communication; and where it does not.
- Implement **gradient saliency**, **Grad-CAM**, **integrated gradients**, **occlusion sensitivity** and **Shapley values** from scratch on a small CNN, and state precisely what each one computes.
- Check the **additivity** property of Shapley values ($f(x) = \phi_0 + \sum_i\phi_i$) and the **completeness** property of integrated gradients numerically.
- Run the **model-randomisation sanity check** (Adebayo et al., 2018) and a localisation check against known object masks.
- Use XAI to uncover a **shortcut** (a spurious feature) that a test-set accuracy figure would hide.
- Explain detections of a **real object detector on KITTI images** (pedestrian and car) with gradient saliency, Grad-CAM and whole-detector occlusion, and run the randomisation sanity check on it.
- Explain why post-hoc explanations are, at best, supporting evidence in an ISO/PAS 8800 assurance argument.

In [ ]:
# Course toolkit (tables, formulas and real-data loaders). Installs automatically on Colab.
try:
    import avsafety
except ImportError:
    %pip install -q "avsafety[data] @ git+https://github.com/milinpatel07/Autonomous-Driving_AI-Safety-and-Security.git"
    import avsafety
from avsafety import data as D

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import spearmanr
from scipy.ndimage import binary_dilation

rng = np.random.default_rng(42)
torch.manual_seed(42)
torch.set_num_threads(1)
plt.rcParams.update({"figure.dpi": 100})
OI = ["#E69F00", "#56B4E9", "#009E73", "#F0E442", "#0072B2", "#D55E00", "#CC79A7", "#000000"]
pd.set_option("display.width", 160)

## 1. Why explainability matters for AV safety, and what it is not

### The problem
Deep perception networks have millions of parameters and highly non-linear mappings; the path from pixels to a detection cannot be traced by inspection. When an engineer, an assessor or an accident investigator asks "why did the system brake here?" or "why was this pedestrian missed?", the network output alone ("score 0.87") does not answer the question.

### Where explanations help
1. **Debugging and safety analysis:** finding *why* a detector fails, e.g. reliance on spurious features (shortcut learning), which is an input to the safety analysis of AI errors in ISO/PAS 8800 and to the search for triggering conditions in ISO 21448.
2. **Dataset work:** revealing biased or leaking features in the data.
3. **Incident investigation and liability:** reconstructing what the perception system relied on.
4. **Communication and trust:** for developers, assessors and, carefully, users.

### What the standards and regulations actually say (as of October 2026)
- **ISO/IEC TR 5469:2024** is an informative Technical Report. It discusses explainability and transparency among properties relevant to AI in functional safety; as a TR it cannot impose requirements.
- **ISO/PAS 8800:2024** addresses AI safety in road vehicles through safety requirements, data, V&V, safety analysis and an assurance argument. Explanation methods can support several of these activities, but post-hoc XAI is not a stand-alone acceptance criterion.
- **EU AI Act:** vehicle AI is high-risk via Article 6(1) and Annex I Section B, but its requirements reach vehicles indirectly through type-approval delegated acts (Article 2(2)); see [01](01_ai_safety_standards.ipynb). The individual right to explanation (Article 86) concerns decisions by deployers of Annex III systems, not vehicle perception.
- **NHTSA Standing General Order 2021-01** requires reporting of crashes involving ADS and Level 2 ADAS within set deadlines (third amendment effective June 2025); it does not require model explanations.
- No AV type-approval regulation that we are aware of mandates post-hoc XAI.

**Key point:** XAI is a valuable engineering and investigation tool with known reliability limits. It is not a legal requirement for AV approval and not proof of correctness.

### Example: the Uber ATG crash in Tempe, Arizona (2018)
According to the NTSB (HAR-19/03), the automated driving system first detected the pedestrian **5.6 s** before impact. Its classification alternated between vehicle, bicycle and "other", and the system **never classified her as a pedestrian**; with each reclassification the object's tracking history was not used for path prediction. A one-second "action suppression" after hazard detection, the deactivated vehicle AEB, and the operator's inattention and safety-management deficiencies were central findings. Explanations of individual classifications might have helped engineers to find the misclassification pattern earlier, but the root causes were design and safety-management failures, which XAI would not have replaced.

## 2. A small, transparent test bed

To study explanation methods we need a case where we *know* the right answer. We generate 32 × 32 grey-scale "scenes" containing a **pedestrian-like** shape (upright body with a head), a **vehicle-like** shape (wide box with wheels), or **no object** (background only), plus sensor noise and clutter, and keep the ground-truth object mask.

The background class matters. A classifier trained only on "pedestrian vs vehicle" can learn that *pedestrian = absence of vehicle features*; it then rates an empty road as a confident pedestrian, and every explanation method will (correctly) point at the background. This closed-world assumption is a classic functional insufficiency of perception classifiers.

> **Illustrative data:** values below are synthetic, chosen for teaching; they are not measurements.

In [ ]:
IMG = 32
yy, xx = np.mgrid[:IMG, :IMG]

def make_scene(cls, r, watermark=False):
    img = 0.2 + 0.15 * r.standard_normal((IMG, IMG))
    for _ in range(2):                                            # clutter blobs
        x0, y0 = r.integers(0, IMG - 4, 2)
        img[y0:y0 + r.integers(2, 4), x0:x0 + r.integers(2, 4)] += 0.4
    mask = np.zeros((IMG, IMG), bool)
    if cls == 1:                                                  # pedestrian: body + head
        h, w = r.integers(12, 17), r.integers(3, 5)
        x0, y0 = r.integers(4, IMG - 4 - w), r.integers(6, IMG - h - 1)
        mask[y0:y0 + h, x0:x0 + w] = True
        mask |= (yy - (y0 - 3)) ** 2 + (xx - (x0 + w // 2)) ** 2 <= 4
    elif cls == 0:                                                # vehicle: body + two wheels
        w, h = r.integers(13, 19), r.integers(5, 8)
        x0, y0 = r.integers(1, IMG - w - 1), r.integers(4, IMG - h - 3)
        mask[y0:y0 + h, x0:x0 + w] = True
        mask[y0 + h:y0 + h + 2, x0 + 1:x0 + 4] = True
        mask[y0 + h:y0 + h + 2, x0 + w - 4:x0 + w - 1] = True
    img[mask] += 0.25                                             # low contrast: a realistic, not trivial, task
    if watermark:
        img[0:3, 0:3] += 1.5                                      # a spurious marker in the corner
    return img.astype(np.float32), mask

def make_set(n, seed, p_mark_ped=0.0, p_mark_veh=0.0):
    r = np.random.default_rng(seed)
    y = r.integers(0, 3, n)                                       # 0 vehicle, 1 pedestrian, 2 background
    imgs, masks = zip(*[make_scene(c, r, watermark=r.random() < (p_mark_ped if c == 1 else p_mark_veh)) for c in y])
    return torch.tensor(np.stack(imgs))[:, None], torch.tensor(y), np.stack(masks)

X_tr, y_tr, _ = make_set(2000, seed=1)
X_te, y_te, M_te = make_set(400, seed=2)
CLASSES = ["vehicle", "pedestrian", "background"]

fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for k, ax in enumerate(axes.ravel()):
    ax.imshow(X_te[k, 0], cmap="gray"); ax.contour(M_te[k], levels=[0.5], colors=[OI[0]], linewidths=1)
    ax.set_title(CLASSES[int(y_te[k])]); ax.axis("off")
plt.suptitle("Synthetic scenes (orange contour = ground-truth object mask)", fontweight="bold")
plt.tight_layout(); plt.show()
print(f"Training images {len(X_tr)}, test images {len(X_te)}, class counts {np.bincount(y_tr.numpy()).tolist()} ({', '.join(CLASSES)})")

In [ ]:
class SmallCNN(nn.Module):
    """Three conv blocks; the last conv layer (8 x 8 feature maps) is the Grad-CAM target."""
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv2d(1, 16, 3, padding=1)
        self.c2 = nn.Conv2d(16, 32, 3, padding=1)
        self.c3 = nn.Conv2d(32, 32, 3, padding=1)
        self.fc = nn.Linear(32, 3)
    def forward(self, x, return_features=False):
        x = F.max_pool2d(F.relu(self.c1(x)), 2)       # 16 x 16
        x = F.max_pool2d(F.relu(self.c2(x)), 2)       # 8 x 8
        feats = F.relu(self.c3(x))                    # 8 x 8 feature maps (no in-place ops)
        logits = self.fc(feats.mean(dim=(2, 3)))      # global average pooling
        return (logits, feats) if return_features else logits

def train_cnn(X, y, epochs=5, seed=0):
    torch.manual_seed(seed)
    net = SmallCNN()
    opt = torch.optim.Adam(net.parameters(), lr=2e-3)
    g = torch.Generator().manual_seed(seed)
    for _ in range(epochs):
        net.train()
        perm = torch.randperm(len(X), generator=g)
        for i in range(0, len(X), 64):
            idx = perm[i:i + 64]
            opt.zero_grad(); F.cross_entropy(net(X[idx]), y[idx]).backward(); opt.step()
    return net.eval()

@torch.no_grad()
def accuracy(net, X, y):
    return (net(X).argmax(1) == y).float().mean().item()

t0 = time.perf_counter()
model = train_cnn(X_tr, y_tr)
print(f"Trained in {time.perf_counter() - t0:.1f} s; test accuracy {accuracy(model, X_te, y_te):.3f}")

## 3. Five explanation methods, implemented from scratch

We explain the **log-odds** of the target class, $f_c(x) = \log\frac{P(c\mid x)}{1-P(c\mid x)} = z_c - \log\sum_{k\ne c} e^{z_k}$. Unlike a single logit, it cannot change without the prediction changing. All perturbation-based methods and baselines use a **background image with the same noise statistics as the scenes** rather than a black or flat grey image, which would itself be far outside the data (try it: the explanations change).

| Method | What it computes | Resolution | Notes |
|---|---|---|---|
| **Gradient saliency** (Simonyan et al., 2014) | $\lvert\partial f_c/\partial x\rvert$, max over channels | Pixel | Local sensitivity, not contribution; noisy; reflects the model's gradient, not causation |
| **Grad-CAM** (Selvaraju et al., 2017) | $\mathrm{ReLU}\big(\sum_k \alpha_k A^k\big)$ with $\alpha_k = $ spatial mean of $\partial f_c/\partial A^k$ | Feature map (here 8 × 8), upsampled | Class-discriminative and less noisy, but **coarse**: lower resolution than gradients. Guided Grad-CAM multiplies it with guided backpropagation for detail |
| **Integrated gradients** (Sundararajan et al., 2017) | $(x - x')\odot\int_0^1 \partial f_c(x' + \alpha(x-x'))/\partial x\, d\alpha$ | Pixel | **Completeness:** attributions sum to $f_c(x) - f_c(x')$ for baseline $x'$ |
| **Occlusion** (Zeiler & Fergus, 2014) | Drop in $f_c$ when a patch is replaced by background | Patch | Perturbation-based, model-agnostic, costly |
| **Shapley values** (Lundberg & Lee, 2017) | Average marginal contribution of each patch over all orderings | Patch (superpixel) | **Local accuracy:** $f(x) = \phi_0 + \sum_i \phi_i$, with $\phi_0$ the value with all patches "absent" |

**LIME** (Ribeiro et al., 2016), not implemented here, fits a sparse linear surrogate $g$ to the model around $x$ by minimising $\sum_z \pi_x(z)\,(f(z) - g(z'))^2 + \Omega(g)$, with a proximity kernel $\pi_x(z) = \exp(-D(x,z)^2/\sigma^2)$ and a complexity penalty $\Omega$. Its explanations depend on the kernel width and the sampling, and can be unstable between runs.

In [ ]:
# Reference 'absence' image: background with the same noise statistics as the scenes (a constant
# grey image would itself be out of distribution for a network trained on noisy images)
BACKGROUND = torch.tensor(0.2 + 0.15 * np.random.default_rng(99).standard_normal((1, 1, IMG, IMG)), dtype=torch.float32)

def margin(logits, c):
    """Log-odds of class c: log P(c) - log(1 - P(c)) = z_c - logsumexp(z_other)."""
    other = torch.cat([logits[:, :c], logits[:, c + 1:]], dim=1)
    return logits[:, c] - torch.logsumexp(other, dim=1)

def saliency(net, x, c):
    x = x.clone().detach().requires_grad_(True)
    net.zero_grad(set_to_none=True)
    margin(net(x), c).sum().backward()
    return x.grad.abs().amax(dim=1).detach()                       # (N, H, W)

def grad_cam(net, x, c):
    logits, feats = net(x, return_features=True)
    grads = torch.autograd.grad(margin(logits, c).sum(), feats)[0] # d f_c / d A^k
    alpha = grads.mean(dim=(2, 3), keepdim=True)                   # GAP of gradients
    cam = F.relu((alpha * feats).sum(dim=1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
    return cam[:, 0].detach()

def integrated_gradients(net, x, c, baseline=None, steps=64):
    baseline = BACKGROUND.expand_as(x) if baseline is None else baseline
    alphas = (torch.arange(steps, dtype=torch.float32) + 0.5) / steps    # midpoint rule
    total = torch.zeros_like(x)
    for a in alphas:
        xi = (baseline + a * (x - baseline)).requires_grad_(True)
        g = torch.autograd.grad(margin(net(xi), c).sum(), xi)[0]
        total += g
    return ((x - baseline) * total / steps)[:, 0].detach()           # signed, (N, H, W)

@torch.no_grad()
def occlusion(net, x, c, patch=6, stride=2, fill=BACKGROUND):
    base = margin(net(x), c)
    heat = torch.zeros(x.shape[0], IMG, IMG); count = torch.zeros(IMG, IMG)
    for i in range(0, IMG - patch + 1, stride):
        for j in range(0, IMG - patch + 1, stride):
            xo = x.clone(); xo[:, :, i:i + patch, j:j + patch] = fill[:, :, i:i + patch, j:j + patch]
            drop = base - margin(net(xo), c)
            heat[:, i:i + patch, j:j + patch] += drop[:, None, None]; count[i:i + patch, j:j + patch] += 1
    return heat / count

@torch.no_grad()
def shapley_patches(net, x, c, grid=4, n_perm=300, fill=BACKGROUND, seed=0):
    """Permutation-sampling Shapley values of grid x grid patches for ONE image x (1, 1, H, W).
    Absent patches are replaced by the background value `fill`. Each permutation's marginal
    contributions sum exactly to f(x) - f(empty), so efficiency holds exactly."""
    r = np.random.default_rng(seed); n = grid * grid; s = IMG // grid
    def compose(present):                                            # present: (B, n) bool
        B = present.shape[0]
        out = fill.expand(B, 1, IMG, IMG).clone()
        for p in range(n):
            i, j = divmod(p, grid)
            sel = torch.as_tensor(present[:, p])
            out[sel, :, i*s:(i+1)*s, j*s:(j+1)*s] = x[0, :, i*s:(i+1)*s, j*s:(j+1)*s]
        return out
    phi = np.zeros(n)
    for _ in range(n_perm):
        order = r.permutation(n)
        present = np.zeros((n + 1, n), bool)
        for k in range(1, n + 1):
            present[k] = present[k - 1]; present[k, order[k - 1]] = True
        v = margin(net(compose(present)), c).numpy()
        phi[order] += np.diff(v)
    phi /= n_perm
    f_empty = margin(net(compose(np.zeros((1, n), bool))), c)[0].item()
    f_full = margin(net(x), c)[0].item()
    return phi.reshape(grid, grid), f_empty, f_full

In [ ]:
# Explain one correctly classified pedestrian with a typical confidence (P closest to 0.9)
with torch.no_grad():
    probs_te = F.softmax(model(X_te), -1)
    pred = probs_te.argmax(1)
cand = np.where((y_te == 1).numpy() & (pred == 1).numpy())[0]
k = int(cand[np.argmin(np.abs(probs_te[cand, 1].numpy() - 0.9))])
x = X_te[k:k + 1]; c = 1
maps = {
    "Gradient saliency": saliency(model, x, c)[0],
    "Grad-CAM (8x8, upsampled)": grad_cam(model, x, c)[0],
    "Integrated gradients": integrated_gradients(model, x, c)[0],
    "Occlusion (drop in log-odds)": occlusion(model, x, c)[0],
}
phi, f_empty, f_full = shapley_patches(model, x, c)

fig, axes = plt.subplots(1, 6, figsize=(21, 4))
axes[0].imshow(x[0, 0], cmap="gray"); axes[0].contour(M_te[k], levels=[0.5], colors=[OI[0]])
p_ped = F.softmax(model(x), -1)[0, 1].item()
axes[0].set_title(f"Input, P(pedestrian) = {p_ped:.3f}")
for ax, (name, m) in zip(axes[1:5], maps.items()):
    m = m.numpy()
    lim = np.abs(m).max()
    ax.imshow(m, cmap="RdBu_r" if m.min() < 0 else "viridis", vmin=-lim if m.min() < 0 else 0, vmax=lim)
    ax.contour(M_te[k], levels=[0.5], colors=[OI[0]], linewidths=1); ax.set_title(name)
lim = np.abs(phi).max()
axes[5].imshow(np.kron(phi, np.ones((IMG // 4, IMG // 4))), cmap="RdBu_r", vmin=-lim, vmax=lim)
axes[5].contour(M_te[k], levels=[0.5], colors=[OI[0]], linewidths=1); axes[5].set_title("Shapley values (4x4 patches)")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

ig = maps["Integrated gradients"]
with torch.no_grad():
    f_x, f_0 = margin(model(x), c)[0].item(), margin(model(BACKGROUND), c)[0].item()
print("Integrated gradients completeness check (log-odds of 'pedestrian', baseline = background image):")
print(f"  sum of attributions = {ig.sum().item():.4f};  f(x) - f(baseline) = {f_x - f_0:.4f}  "
      f"(relative error {abs(ig.sum().item() - (f_x - f_0)) / abs(f_x - f_0):.2%}, from the 64-step integral)")
print("Shapley local accuracy check (log-odds of 'pedestrian', absent patches = background):")
print(f"  phi_0 = f(all patches absent) = {f_empty:.4f};  sum phi_i = {phi.sum():.4f};  "
      f"phi_0 + sum phi_i = {f_empty + phi.sum():.4f};  f(x) = {f_full:.4f}")
occ = maps["Occlusion (drop in log-odds)"].numpy()
print(f"Occlusion: mean drop inside the mask {occ[M_te[k]].mean():+.3f}, outside {occ[~M_te[k]].mean():+.3f}"
      + ("  <- partial occlusion RAISES the score here: patch interactions make perturbation maps disagree with gradients"
         if occ[M_te[k]].mean() < 0 else ""))
top = np.argsort(-phi.ravel())[:3]
print("  Largest patch contributions: " + ", ".join(f"patch (row {t // 4}, col {t % 4}) {phi.ravel()[t]:+.3f}" for t in top))

Different methods answer different questions: gradients describe infinitesimal sensitivity, occlusion the effect of removing one patch while the rest stays, Shapley values average over all combinations of present and absent patches. When patches interact, these answers can disagree, even in sign; comparing several methods is therefore more informative than trusting one.

Note on the Shapley "base value" $\phi_0$: here it is the model output with all patches replaced by background. With SHAP's `KernelExplainer`, $\phi_0$ is the *expected* output over a background data set. Either way, attributions sum to $f(x) - \phi_0$, **not** to $f(x)$. A statement such as "0.87 = 0.3 (shape) + 0.4 (colour) + 0.17 (context)" omits the base value; and for images, the "features" are pixels or superpixels, not abstract concepts like shape or colour.

## 4. Do the explanations point at the object? A localisation check

Because the object masks are known, we can measure the share of positive attribution inside the mask and compare it with the share of the image the mask covers (what a random map would achieve).

In [ ]:
idx = np.where((y_te == 1).numpy() & (pred == 1).numpy())[0][:60]
Xs = X_te[idx]; Ms = M_te[idx]
batch_maps = {
    "Gradient saliency": saliency(model, Xs, 1),
    "Grad-CAM": grad_cam(model, Xs, 1),
    "Integrated gradients": integrated_gradients(model, Xs, 1).clamp_min(0),
    "Occlusion": occlusion(model, Xs, 1).clamp_min(0),
}
area = Ms.mean(axis=(1, 2))
Md = np.stack([binary_dilation(mk, iterations=4) for mk in Ms])          # mask grown by one feature-map cell (4 px)
area_d = Md.mean(axis=(1, 2))
share = lambda m, msk: (m.numpy() * msk).sum(axis=(1, 2)) / (m.numpy().sum(axis=(1, 2)) + 1e-12)
loc_df = pd.DataFrame({name: [share(m, Ms).mean(), (share(m, Ms) / area).mean(),
                              share(m, Md).mean(), (share(m, Md) / area_d).mean()]
                       for name, m in batch_maps.items()},
                      index=["share inside mask", "ratio to mask area (random = 1)",
                             "share inside mask + 4 px", "ratio, mask + 4 px"]).T
print(f"Mean mask area: {area.mean():.1%} of the image; grown by 4 px: {area_d.mean():.1%} ({len(idx)} pedestrian images)")
print(loc_df.round(3).to_string())
best = loc_df.iloc[:, 0].idxmax()
print(f"\nMost concentrated on the object: {best}.")
for name, r in loc_df.iterrows():
    print(f"  {name:22s}: {r.iloc[1]:.1f}x random inside the mask, {r.iloc[3]:.1f}x random inside the mask grown by 4 px")

Grad-CAM is computed on 8 × 8 feature maps, so one cell covers 4 × 4 pixels, while the pedestrian body is only 3–4 pixels wide: after upsampling, part of its mass necessarily falls just outside the exact mask. The columns for the mask grown by one feature-map cell separate this resolution effect from genuine mislocalisation. If Grad-CAM stays near the random level even then, resolution is not the only reason: with global average pooling, class evidence can also come from feature-map cells that respond to context (for example, the absence of vehicle-like structure). Different methods can thus give different "explanations" for the same prediction.

Concentration on the object is *plausibility*, not *faithfulness*: a map can look right and still not reflect what the model uses. That is what the next check targets.

## 5. Sanity check: model-parameter randomisation (Adebayo et al., 2018)

If an explanation method is faithful to the model, then destroying the model's learned weights must change the explanation. In the **cascading randomisation** test, layers are re-initialised from the top (output) down; after each step we compare the new maps with the original ones by Spearman rank correlation (of absolute values). A method whose maps stay similar for a randomised model is explaining the *input* (e.g. edges), not the model. Adebayo et al. found that some popular methods, e.g. guided backpropagation, largely fail this test, while plain gradients and Grad-CAM pass it.

In [ ]:
import copy

def explain_all(net, X):
    return {"Gradient saliency": saliency(net, X, 1).abs(),
            "Grad-CAM": grad_cam(net, X, 1).abs(),
            "Integrated gradients": integrated_gradients(net, X, 1, steps=32).abs()}

def rank_corr(a, b):
    """Spearman correlation; a constant map (e.g. an all-zero Grad-CAM) carries no ranking, so count it as 0."""
    if np.ptp(a) == 0 or np.ptp(b) == 0:
        return 0.0
    return spearmanr(a.ravel(), b.ravel())[0]

def mean_spearman(A_, B_):
    return float(np.mean([rank_corr(a, b) for a, b in zip(A_.numpy(), B_.numpy())]))

Xs40 = Xs[:40]
reference = explain_all(model, Xs40)
stages = ["fc", "c3", "c2", "c1"]
rand_net = copy.deepcopy(model)
torch.manual_seed(123)
rows = [{"randomised up to": "none (trained)", **{k: 1.0 for k in reference}}]
for layer in stages:
    getattr(rand_net, layer).reset_parameters()                    # cascading: keep earlier randomisations
    maps_r = explain_all(rand_net, Xs40)
    rows.append({"randomised up to": layer, **{k: mean_spearman(reference[k], maps_r[k]) for k in reference}})
sanity = pd.DataFrame(rows).set_index("randomised up to")
print("Mean Spearman rank correlation between maps of the trained and the randomised model")
print(sanity.round(3).to_string())

fig, axes = plt.subplots(3, len(stages) + 1, figsize=(15, 9))
img_k = 0
net_c = copy.deepcopy(model); torch.manual_seed(123)
for col, layer in enumerate(["none"] + stages):
    if layer != "none":
        getattr(net_c, layer).reset_parameters()
    mm = explain_all(net_c, Xs40[img_k:img_k + 1])
    for row, name in enumerate(mm):
        axes[row, col].imshow(mm[name][0], cmap="viridis"); axes[row, col].axis("off")
        if col == 0: axes[row, col].set_title(f"{name}\ntrained model", fontsize=9)
        else: axes[row, col].set_title(f"{name}\nrandomised to {layer}", fontsize=9)
plt.suptitle("Cascading model randomisation: faithful explanations should degrade", fontweight="bold")
plt.tight_layout(); plt.show()

final = sanity.iloc[-1]
for name, v in final.items():
    verdict = ("maps change substantially (passes)" if abs(v) < 0.3 else
               "maps partly persist: partly driven by the input itself" if abs(v) < 0.6 else
               "WARNING: maps remain similar to the trained model's (fails)")
    print(f"{name:22s}: correlation after full randomisation {v:+.2f} -> {verdict}")

Even a method that "passes" can retain some correlation after randomisation, because the input image itself (bright object on a dark background) shapes the gradients of any convolutional network. Integrated gradients multiplies gradients by $(x - x')$, so its maps inherit structure from the input image, which is why it keeps more correlation than plain gradients here; Adebayo et al. report the same tendency for gradient ⊙ input. A constant map (e.g. Grad-CAM that becomes all zero after the output layer is randomised) is counted as correlation 0. Adebayo et al. also propose a **data randomisation test** (retrain on permuted labels); both tests are necessary conditions for faithfulness, not proofs of it.

## 6. Using XAI in safety analysis: uncovering a shortcut

Suppose a data-collection artefact correlates with the class: in the training data, 95 % of pedestrian images carry a small bright marker in the top-left corner (think of a timestamp overlay used only on one recording campaign), and 5 % of the other images do. A network trained on such data may learn the marker instead of the pedestrian.

Accuracy on a test set **with the same artefact** can look excellent. Explanations reveal the problem, and a targeted test confirms it.

In [ ]:
X_trs, y_trs, _ = make_set(2000, seed=1, p_mark_ped=0.95, p_mark_veh=0.05)
shortcut_model = train_cnn(X_trs, y_trs, seed=0)

X_same, y_same, _ = make_set(400, seed=3, p_mark_ped=0.95, p_mark_veh=0.05)   # same artefact as training
X_clean, y_clean, M_clean = make_set(400, seed=4)                             # no marker at all
X_swap, y_swap, _ = make_set(400, seed=5, p_mark_ped=0.0, p_mark_veh=1.0)    # marker only on non-pedestrians

acc = pd.DataFrame({
    "clean model": [accuracy(model, X_same, y_same), accuracy(model, X_clean, y_clean), accuracy(model, X_swap, y_swap)],
    "shortcut model": [accuracy(shortcut_model, X_same, y_same), accuracy(shortcut_model, X_clean, y_clean),
                       accuracy(shortcut_model, X_swap, y_swap)]},
    index=["test with same artefact", "clean test (no marker)", "marker only on non-pedestrians"])
print("Test accuracy"); print(acc.round(3).to_string())

corner = np.zeros((IMG, IMG), bool); corner[0:4, 0:4] = True
ped = (y_same == 1).numpy()
attr = {name: integrated_gradients(net, X_same[ped][:60], 1).clamp_min(0).numpy()
        for name, net in [("clean model", model), ("shortcut model", shortcut_model)]}
share = {name: float(((a * corner).sum(axis=(1, 2)) / (a.sum(axis=(1, 2)) + 1e-12)).mean()) for name, a in attr.items()}

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
j = 0
axes[0].imshow(X_same[ped][j, 0], cmap="gray"); axes[0].set_title("Pedestrian image with marker")
for ax, name in zip(axes[1:], attr):
    ax.imshow(attr[name][j], cmap="viridis"); ax.set_title(f"{name}: IG (positive part)\ncorner share {share[name]:.1%}")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

print(f"\nShare of positive IG attribution in the 4x4 corner (1.6 % of the image): "
      f"clean model {share['clean model']:.1%}, shortcut model {share['shortcut model']:.1%}")
drop = acc.loc["test with same artefact", "shortcut model"] - acc.loc["marker only on non-pedestrians", "shortcut model"]
print(f"Shortcut model: accuracy falls by {drop:.1%} when the marker no longer correlates with the class.")

## 7. Real images, real detector: explaining Faster R-CNN detections on KITTI

The test bed above showed the methods where the right answer is known. A production perception stack is a **detector**, not a classifier, and it sees real images. We now explain detections of torchvision's Faster R-CNN with a MobileNetV3-Large FPN backbone (COCO-trained, not fine-tuned; Ren et al., 2017) on two KITTI frames: a pedestrian and a car. A detector's output has several stages (backbone, region proposals, box classification and regression, non-maximum suppression), so "what does the model look at?" has to be made precise. We use two complementary views:

- **White-box, fixed box.** Hold the detected box fixed, run it through the box classifier (RoI head), and differentiate the log-odds of its class with respect to the input pixels (**gradient saliency**) and to the deepest backbone feature map (**Grad-CAM**; 960 channels on a 13 × 42 grid, one cell ≈ 30 × 30 image pixels). This explains the *classification of that box* only; how the proposal arose, the box regression and NMS are not covered.
- **Black-box, whole detector.** In the spirit of D-RISE (Petsiuk et al., 2021), occlude a patch around the object (replaced by a strongly blurred copy of the image, which stays closer to natural image statistics than a grey square), re-run the **complete detector**, and record the drop in the similarity $\max_j \mathrm{IoU}(b_j, b^\star)\, s_j$ between the target box $b^\star$ and the detections $b_j$ of the same class. D-RISE uses random masks; a regular 6 × 6 grid of overlapping patches is used here to keep it to 36 detector runs per object.

### Data and licence

Two frames of the KITTI object-detection training set (Geiger et al., 2012), licence CC BY-NC-SA 3.0 (non-commercial use), fetched on demand by `avsafety.data` and cached; the same frames are used for calibration in [04](04_calibration.ipynb). The detector weights (about 75 MB) are downloaded once by torchvision.

In [ ]:
import os
import torchvision
from torchvision.transforms.functional import gaussian_blur

print(D.attribution("KITTI"))
torch.set_num_threads(min(4, os.cpu_count()))
detector = torchvision.models.detection.fasterrcnn_mobilenet_v3_large_fpn(weights="DEFAULT").eval()
for p in detector.parameters():
    p.requires_grad_(False)                                 # gradients are needed w.r.t. inputs and features only
CASES = {"pedestrian": dict(frame=1428, coco=1, kitti="Pedestrian", iou=0.5),   # COCO 1 = person
         "car": dict(frame=2444, coco=3, kitti="Car", iou=0.7)}                 # COCO 3 = car

def iou_one(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    iw = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); ih = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter)

for name, case in CASES.items():
    try:
        fr = D.kitti_object(case["frame"], parts=("image", "labels"))
    except D.DataUnavailable as exc:
        raise SystemExit(f"KITTI could not be downloaded ({exc}). Run the notebook with internet access.")
    x = torch.from_numpy(fr.image.copy()).permute(2, 0, 1).float() / 255
    with torch.no_grad():
        out = detector([x])[0]
    gts = [l for l in fr.labels if l["type"] == case["kitti"] and D.kitti_difficulty(l) == "Easy"]
    best = None                                            # highest-scoring detection that matches an Easy object
    for b, s, lab in zip(out["boxes"].numpy(), out["scores"].numpy(), out["labels"].numpy()):
        if lab != case["coco"]:
            continue
        ious = [iou_one(b, [g["x1"], g["y1"], g["x2"], g["y2"]]) for g in gts]
        if ious and max(ious) >= case["iou"] and (best is None or s > best[1]):
            best = (b, float(s), gts[int(np.argmax(ious))], max(ious))
    case.update(image=fr.image, x=x, box=torch.tensor(best[0]), score=best[1], gt=best[2], iou_gt=best[3])
    print(f"{name:10s}: frame {case['frame']:06d}, detection score {best[1]:.3f}, IoU with ground truth {best[3]:.2f}, "
          f"box height {best[0][3] - best[0][1]:.0f} px, ground-truth distance {np.hypot(best[2]['x'], best[2]['z']):.1f} m")

In [ ]:
def box_log_odds(net, x, box, c):
    """Log-odds of class c for a FIXED box, through transform, backbone, FPN and RoI head.
    Returns the log-odds and the deepest backbone feature map (for Grad-CAM)."""
    images, _ = net.transform([x])
    body = net.backbone.body(images.tensors)               # {'0': stride-16 features, '1': deepest features}
    feats = net.backbone.fpn(body)
    (h0, w0), (h1, w1) = x.shape[-2:], images.image_sizes[0]
    b = (box * torch.tensor([w1 / w0, h1 / h0, w1 / w0, h1 / h0]))[None]
    pooled = net.roi_heads.box_roi_pool(feats, [b], images.image_sizes)
    z, _ = net.roi_heads.box_predictor(net.roi_heads.box_head(pooled))
    z = z[0]
    return z[c] - torch.logsumexp(torch.cat([z[:c], z[c + 1:]]), 0), body["1"], (h1, w1)

def saliency_and_gradcam(net, case):
    x = case["x"].clone().requires_grad_(True)
    f, A, (h1, w1) = box_log_odds(net, x, case["box"], case["coco"])
    gx, gA = torch.autograd.grad(f, [x, A])
    sal = gx.abs().amax(0)                                                   # (H, W) in original pixels
    cam = F.relu((gA.mean(dim=(2, 3), keepdim=True) * A).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=(h1, w1), mode="bilinear", align_corners=False)
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
    return sal.detach().numpy(), cam.detach().numpy(), f.item()

def window_of(case, grow_w=0.5, grow_h=0.25):
    x1, y1, x2, y2 = case["box"].tolist(); w, h = x2 - x1, y2 - y1
    H, W = case["image"].shape[:2]
    return (int(max(0, x1 - grow_w * w)), int(max(0, y1 - grow_h * h)),
            int(min(W, x2 + grow_w * w)), int(min(H, y2 + grow_h * h)))

@torch.no_grad()
def occlusion_detector(case, grid=6, batch=4):
    """Drop in max_j IoU(b_j, b*) * s_j of the whole detector when a patch is replaced by a blurred copy."""
    x, target, c = case["x"], case["box"].numpy(), case["coco"]
    blurred = gaussian_blur(x, kernel_size=51, sigma=15.0)
    X0, Y0, X1, Y1 = window_of(case)
    pw, ph = (X1 - X0) / 3, (Y1 - Y0) / 3                                     # patch = 1/3 of the window
    xs, ys = np.linspace(X0, X1 - pw, grid), np.linspace(Y0, Y1 - ph, grid)
    def similarity(outs):
        return [max([iou_one(b, target) * s for b, s, l in zip(o["boxes"].numpy(), o["scores"].numpy(),
                                                                 o["labels"].numpy()) if l == c] + [0.0]) for o in outs]
    s0 = similarity(detector([x]))[0]
    patches = [(int(px), int(py)) for py in ys for px in xs]
    sims = []
    for i in range(0, len(patches), batch):
        imgs = []
        for px, py in patches[i:i + batch]:
            xo = x.clone()
            xo[:, py:py + int(ph), px:px + int(pw)] = blurred[:, py:py + int(ph), px:px + int(pw)]
            imgs.append(xo)
        sims += similarity(detector(imgs))
    heat = np.zeros(x.shape[-2:]); count = np.zeros(x.shape[-2:])
    for (px, py), s in zip(patches, sims):
        heat[py:py + int(ph), px:px + int(pw)] += s0 - s; count[py:py + int(ph), px:px + int(pw)] += 1
    return np.where(count > 0, heat / np.maximum(count, 1), np.nan), s0, np.array(sims)

t0 = time.perf_counter()
for case in CASES.values():
    case["sal"], case["cam"], case["log_odds"] = saliency_and_gradcam(detector, case)
    case["occ"], case["sim0"], case["sims"] = occlusion_detector(case)
print(f"Explanations computed in {time.perf_counter() - t0:.0f} s "
      f"({sum(len(c['sims']) + 1 for c in CASES.values())} full detector runs for occlusion)")

In [ ]:
def in_box_mask(case, shape):
    g = case["gt"]; m = np.zeros(shape, bool)
    m[int(g["y1"]):int(np.ceil(g["y2"])), int(g["x1"]):int(np.ceil(g["x2"]))] = True
    return m

METHODS = [("sal", "Gradient saliency (box classifier)"), ("cam", "Grad-CAM (deepest backbone layer)"),
           ("occ", "Occlusion of the whole detector")]
fig, axes = plt.subplots(2, 4, figsize=(18, 9), gridspec_kw={"width_ratios": [1, 1, 1, 1]})
rows = []
for r, (name, case) in enumerate(CASES.items()):
    X0, Y0, X1, Y1 = window_of(case)
    crop = case["image"][Y0:Y1, X0:X1]
    g, (bx1, by1, bx2, by2) = case["gt"], case["box"].tolist()
    def outline(ax):
        ax.add_patch(plt.Rectangle((g["x1"] - X0, g["y1"] - Y0), g["x2"] - g["x1"], g["y2"] - g["y1"],
                                   fill=False, ec=OI[2], lw=2, ls="--"))
        ax.add_patch(plt.Rectangle((bx1 - X0, by1 - Y0), bx2 - bx1, by2 - by1, fill=False, ec="white", lw=1.2))
    axes[r, 0].imshow(crop); outline(axes[r, 0])
    axes[r, 0].set_title(f"{name}: score {case['score']:.3f}\n(green dashed: ground truth, white: detection)", fontsize=10)
    mask_full = in_box_mask(case, case["image"].shape[:2])[Y0:Y1, X0:X1]
    for col, (key, title) in enumerate(METHODS, start=1):
        m = case[key][Y0:Y1, X0:X1]
        ax = axes[r, col]
        ax.imshow(crop.mean(axis=2), cmap="gray", vmin=0, vmax=255)
        if key == "occ":
            lim = np.nanmax(np.abs(m))
            im = ax.imshow(m, cmap="RdBu_r", vmin=-lim, vmax=lim, alpha=0.65)
        else:
            im = ax.imshow(m, cmap="viridis", alpha=0.65)
        outline(ax); ax.set_title(title, fontsize=10)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02,
                     label="drop in IoU x score" if key == "occ" else "attribution (a.u.)")
        pos = np.nan_to_num(np.clip(m, 0, None))
        share = pos[mask_full].sum() / (pos.sum() + 1e-12)
        thirds = np.array_split(np.arange(int(g["y1"]) - Y0, int(np.ceil(g["y2"])) - Y0), 3)
        inside = [pos[t][:, int(g["x1"]) - X0:int(np.ceil(g["x2"])) - X0].sum() for t in thirds]
        rows.append(dict(object=name, method=title.split(" (")[0], share_in_gt_box=share,
                         ratio_to_area=share / mask_full.mean(),
                         top_third=inside[0] / sum(inside), middle_third=inside[1] / sum(inside),
                         bottom_third=inside[2] / sum(inside)))
for ax in axes.ravel():
    ax.axis("off")
plt.suptitle("What drives each detection? Window around the object; heat maps over the greyscale image", fontweight="bold")
plt.tight_layout(); plt.show()

loc = pd.DataFrame(rows).set_index(["object", "method"])
print("Positive attribution in the analysis window: share inside the ground-truth box, ratio to the box's share of the\n"
      "window (1 = no better than uniform), and how the in-box share splits over the box's top/middle/bottom third")
print(loc.round(2).to_string())
for name, case in CASES.items():
    print(f"{name:10s}: unoccluded similarity {case['sim0']:.3f}; strongest single-patch drop "
          f"{case['sim0'] - case['sims'].min():.3f}; patches that RAISE the similarity: {(case['sims'] > case['sim0']).sum()} of {len(case['sims'])}")

How to read this: the three maps answer different questions, as in Section 3. Gradient saliency is the local sensitivity of the box classifier's log-odds to each pixel; Grad-CAM is a coarse class-weighted map of the deepest features (cells of about 30 px, so it cannot resolve a pedestrian's limbs); occlusion is the effect of removing a patch on the *whole* detector, including whether a box is still proposed at the same place. The ratio column compares each method's concentration on the object with a uniform map, and the thirds show which part of the object each method emphasises.

### 7.1 Sanity check on the real detector

The cascading randomisation test of Section 5 applies unchanged: re-initialise the box predictor, then the two fully connected layers of the RoI head, then the FPN, and compare the maps with those of the trained detector (Spearman correlation of absolute values over the analysis window). Occlusion is not repeated here, because 36 detector runs per stage would dominate the run time; as a black-box method it depends on the model only through its outputs and is checked differently (e.g. by deletion/insertion curves).

In [ ]:
import copy
def window_maps(net, case):
    X0, Y0, X1, Y1 = window_of(case)
    s, c, _ = saliency_and_gradcam(net, case)
    return s[Y0:Y1, X0:X1], c[Y0:Y1, X0:X1]

STAGES = [("box predictor", lambda n: [n.roi_heads.box_predictor]),
          ("+ RoI head (fc6, fc7)", lambda n: [n.roi_heads.box_head]),
          ("+ FPN", lambda n: [n.backbone.fpn])]
torch.manual_seed(123)
rand_net = copy.deepcopy(detector)
rows = []
ref = {name: window_maps(detector, case) for name, case in CASES.items()}
for stage, modules in STAGES:
    for mod in modules(rand_net):
        for sub in mod.modules():
            if hasattr(sub, "reset_parameters"):
                sub.reset_parameters()                        # cascading: earlier randomisations are kept
    for name, case in CASES.items():
        s_r, c_r = window_maps(rand_net, case)
        rows.append({"randomised": stage, "object": name,
                     "Gradient saliency": rank_corr(np.abs(ref[name][0]), np.abs(s_r)),
                     "Grad-CAM": rank_corr(np.abs(ref[name][1]), np.abs(c_r))})
sanity_real = pd.DataFrame(rows).set_index(["randomised", "object"])
print("Spearman correlation with the trained detector's maps (1 = unchanged)")
print(sanity_real.round(2).to_string())
final = sanity_real.xs(STAGES[-1][0], level="randomised").mean()
for method, v in final.items():
    print(f"{method:18s}: mean correlation after randomising predictor, RoI head and FPN {v:+.2f} -> "
          + ("maps change substantially (passes)" if abs(v) < 0.3 else
             "maps partly persist: driven partly by the input and the untouched backbone" if abs(v) < 0.6 else
             "maps remain similar (fails)"))

The randomisation stops at the FPN, so the MobileNet backbone keeps its trained weights in every stage. Correlation that persists after the last stage therefore reflects both the input image and the features of the untouched backbone; randomising the backbone too (an exercise) separates the two.

### 7.2 What the real-image explanations do and do not establish

- They show **where** the detection is sensitive for **this** image, with method-dependent answers. That is useful for a hypothesis ("the pedestrian score depends mostly on the upper body", "the car score depends on context to the side of the box"), which must then be tested on a targeted data slice, as in Section 6.
- They do **not** show that the detector is correct, robust, or that it uses the "right" features in general: one explanation covers one input, and a plausible map is not evidence of faithfulness.
- For a detector, the white-box maps explain only the box classifier for a fixed box; the black-box map depends on the fill, the patch size and the similarity measure. Both must be fixed and justified before they are used in a safety analysis.
- Explanations of *missed* objects are the most relevant for AEB (SG-02) and the hardest to compute: there is no output box to explain, so one explains the score of the ground-truth box under the box classifier, or the proposals around it.

In ISO/PAS 8800 terms, the explanation pointed to a **data-related insufficiency** (a spurious correlation in the dataset) that caused an **AI error** under a specific condition. The *evidence* that the problem exists, and later that it is fixed, comes from the targeted test sets, not from the heat map itself. That is the appropriate role of XAI: a hypothesis generator for safety analysis, followed by verification.

## 8. Limits of XAI as safety evidence (ISO/PAS 8800 perspective)

| Limitation | Consequence for an assurance argument |
|---|---|
| **Unfaithfulness:** maps may reflect the input or the method more than the model (Adebayo et al., 2018) | Any explanation method used as evidence needs its own sanity checks and faithfulness metrics |
| **Manipulability:** explanations can be changed by imperceptible input perturbations or by adversarially fine-tuned models while predictions stay the same (Dombrowski et al., 2019) | Explanations are not robust evidence under adversarial conditions; relevant for ISO/SAE 21434 as well |
| **Method dependence:** different methods (and baselines, kernel widths, patch sizes) give different answers | Fix and justify the method and its parameters; report variability |
| **Confirmation bias:** a plausible-looking map invites over-trust | Use explanations to generate hypotheses, then test them with data |
| **Local, not global:** one explanation covers one input | Statements about the ODD need aggregated, statistical evidence per slice |
| **Detectors and fusion:** most methods were designed for classifiers; for detectors the target is a box's class logit, and fused or temporal models complicate attribution | Explanations for production perception stacks are research topics, not mature tools |
| **Attention is not explanation:** attention weights show information routing, not necessarily causal importance (Jain & Wallace, 2019; Wiegreffe & Pinter, 2019) | Use gradient-weighted attention or rollout with care |

**Bottom line:** in an ISO/PAS 8800 assurance argument, explanations can support claims about the *absence of known shortcut features* or the *plausibility of learned features*, provided the method passed sanity checks and the claim is confirmed by targeted tests. They do not replace statistical V&V, safety analysis or field monitoring.

## 9. Documenting explanations for an AI component

The template below is **illustrative**. All numbers are placeholders, not results, and acceptance criteria come from the project's AI safety requirements and SOTIF validation targets, not from ASIL values (ISO 26262 ASILs do not define perception error rates).

In [ ]:
documentation_template = """
================================================================
AI COMPONENT DOCUMENTATION - EXPLAINABILITY SECTION
(illustrative; informed by ISO/PAS 8800:2024 and ISO/IEC TR 5469:2024)
================================================================
COMPONENT : Pedestrian classifier, camera channel      VERSION: <x.y.z>
AUTHOR    : <name>                                     DATE   : <date>

1. EXPLANATION METHODS AND PARAMETERS
   Primary  : Integrated gradients (target: log-odds; baseline: background image, 64 steps)
   Secondary: Grad-CAM on last conv block (8x8, bilinear upsampling)
   Sanity checks: model-randomisation test, Spearman correlation
                  after full randomisation: IG <value>, Grad-CAM <value>

2. INTENDED USE OF EXPLANATIONS
   - Safety analysis: search for shortcut features per ODD slice
   - Incident investigation: reconstruction of attribution for logged frames
   - NOT used as acceptance evidence on their own

3. FINDINGS (example)
   F-01: Attribution concentrated on corner marker in campaign C-07 data
         -> dataset insufficiency; targeted test set T-SC-01 created
         -> accuracy with marker decorrelated: <value> (before fix) / <value> (after fix)

4. ACCEPTANCE CRITERIA (from AI safety requirements, per ODD slice)
   FNR <= <X> in slice <Y> with 95 % confidence  -> status: <met / not met>

5. LIMITATIONS
   Explanations are local, method-dependent and can be manipulated;
   see Section 8 of the teaching notebook.
================================================================
"""
print(documentation_template)

## 10. Accuracy vs interpretability

- **Inherently interpretable models** (linear models, small trees, prototype networks) are easier to explain but often insufficient for camera or LiDAR perception.
- **Complex models with post-hoc explanations** keep performance, but the explanations themselves must be validated.
- **Architectural safety measures** often matter more than either: an interpretable *monitor* or rule-based plausibility check next to a complex detector (e.g. a LiDAR free-space check that can veto a "no obstacle" output) gives a safety argument that does not rely on explaining the network.

Public evidence on how AV developers use XAI in production is limited; published safety reports focus on safety cases, scenario-based testing and field performance rather than on XAI.

## In industry

**Where explanations are used.** In practice XAI is an **engineering and analysis tool**, not an acceptance criterion. It feeds three work products: the **safety analysis of AI errors** (ISO/PAS 8800; hypotheses about the causes of misses and false detections, e.g. shortcut features or reliance on context), the **SOTIF triggering-condition analysis** (ISO 21448; a hypothesis such as "glare on the upper body suppresses the pedestrian score" becomes a candidate triggering condition to be tested), and the **investigation of field events** (what the perception output depended on in a logged frame). Explanations can also support dataset reviews by showing label or recording artefacts.

**Who produces and who reads it.** Perception ML engineers produce the explanations; safety analysts and SOTIF engineers turn them into hypotheses and test cases; incident investigators use them on logged data; assessors review the method, not the pictures.

**What reviewers and assessors check**

- The method, its target (which output, which class, which box) and its parameters (baseline or fill, patch size, layer) are fixed, documented and versioned with the model.
- Sanity checks (model and data randomisation) and a faithfulness check (e.g. deletion/insertion curves) have been passed for this model, not only in the literature.
- Every safety-relevant conclusion drawn from an explanation is backed by a **targeted test** on a data slice, and the test, not the heat map, is the evidence.
- Explanations are not used to argue correctness, robustness or ODD coverage; limitations (local, method-dependent, manipulable) are stated.
- If explanation tooling is used to produce safety evidence, it falls under the tool-confidence considerations of ISO/PAS 8800 like any other analysis tool.

## Key takeaways

- XAI supports debugging, safety analysis and investigation; **no AV regulation mandates post-hoc XAI**, and ISO/IEC TR 5469 is informative.
- Know what each method computes: saliency = local gradient; Grad-CAM = coarse, class-discriminative feature-map weighting; integrated gradients = attributions summing to $f(x) - f(x')$; Shapley values sum to $f(x) - \phi_0$, **not to $f(x)$**.
- **Sanity checks** (model and data randomisation) and **localisation/faithfulness checks** are prerequisites before explanations are used as evidence.
- On a **real detector**, "what the model looks at" must name the output being explained: white-box maps for a fixed box explain the box classifier only, black-box occlusion explains the whole detector but depends on fill and patch size; the methods give different answers for the same detection.
- XAI is most useful as a **hypothesis generator** (e.g. a shortcut feature) whose findings are then confirmed with targeted tests.
- In an ISO/PAS 8800 assurance argument, explanations are supporting evidence at best.

## Self-assessment

1. Why does Grad-CAM have lower spatial resolution than gradient saliency?
2. A Shapley explanation lists contributions +0.30, +0.15 and +0.07 for a detection score of 0.87. What is the base value?
3. What does the model-randomisation test check, and what does passing it not prove?
4. How can XAI contribute to the safety analysis of AI errors in ISO/PAS 8800?
5. Why is "the explanation satisfies ISO/IEC TR 5469" not a meaningful compliance statement?
6. For a Faster R-CNN detection, what does a gradient map computed for a fixed box not explain?

<details>
<summary>Answers</summary>

1. Grad-CAM weights the feature maps of a chosen convolutional layer, whose spatial size is a fraction of the input (8 × 8 here, often 7 × 7 or 14 × 14 in large CNNs); the map is then upsampled. Gradient saliency is computed per input pixel.
2. By local accuracy $f(x) = \phi_0 + \sum\phi_i$: $\phi_0 = 0.87 - 0.52 = 0.35$.
3. It checks that explanations depend on the learned parameters: if maps barely change when the weights are randomised, the method is not explaining the model. Passing is a necessary, not a sufficient condition for faithfulness.
4. By generating hypotheses about why errors occur (spurious features, reliance on context, sensitivity to particular conditions), which are then confirmed with targeted test sets and fed back into dataset requirements, architecture or ODD restrictions.
5. A Technical Report is informative and contains no requirements to comply with; compliance statements need a normative reference and an assessment of the full set of work products.
6. How the box was proposed (region proposal network), how it was regressed to its final position, and whether non-maximum suppression kept it. It explains only the class score of that box under the RoI classifier; a whole-detector method such as occlusion or D-RISE is needed for the full pipeline.
</details>

## References

- Selvaraju, R. R. et al. (2017). Grad-CAM: Visual explanations from deep networks via gradient-based localization. ICCV 2017. https://doi.org/10.1109/ICCV.2017.74
- Simonyan, K., Vedaldi, A., Zisserman, A. (2014). Deep inside convolutional networks: visualising image classification models and saliency maps. ICLR Workshop 2014. arXiv:1312.6034
- Sundararajan, M., Taly, A., Yan, Q. (2017). Axiomatic attribution for deep networks. ICML 2017. arXiv:1703.01365
- Zeiler, M. D., Fergus, R. (2014). Visualizing and understanding convolutional networks. ECCV 2014. arXiv:1311.2901
- Lundberg, S. M., Lee, S.-I. (2017). A unified approach to interpreting model predictions. NeurIPS 2017. arXiv:1705.07874
- Ribeiro, M. T., Singh, S., Guestrin, C. (2016). "Why should I trust you?" Explaining the predictions of any classifier. KDD 2016. arXiv:1602.04938
- Adebayo, J. et al. (2018). Sanity checks for saliency maps. NeurIPS 2018. arXiv:1810.03292
- Dombrowski, A.-K. et al. (2019). Explanations can be manipulated and geometry is to blame. NeurIPS 2019. arXiv:1906.07983
- Jain, S., Wallace, B. C. (2019). Attention is not explanation. NAACL 2019. arXiv:1902.10186
- Wiegreffe, S., Pinter, Y. (2019). Attention is not not explanation. EMNLP 2019. arXiv:1908.04626
- Petsiuk, V. et al. (2021). Black-box explanation of object detectors via saliency maps (D-RISE). CVPR 2021. arXiv:2006.03204
- Ren, S., He, K., Girshick, R., Sun, J. (2017). Faster R-CNN: Towards real-time object detection with region proposal networks. *IEEE TPAMI* 39(6), 1137–1149. https://doi.org/10.1109/TPAMI.2016.2577031
- Geiger, A., Lenz, P., Urtasun, R. (2012). Are we ready for autonomous driving? The KITTI vision benchmark suite. CVPR 2012. https://doi.org/10.1109/CVPR.2012.6248074
- Geirhos, R. et al. (2020). Shortcut learning in deep neural networks. *Nature Machine Intelligence* 2, 665–673. https://doi.org/10.1038/s42256-020-00257-z
- NTSB (2019). Collision Between Vehicle Controlled by Developmental Automated Driving System and Pedestrian, Tempe, Arizona, March 18, 2018. Highway Accident Report NTSB/HAR-19/03. https://www.ntsb.gov/investigations/Pages/HWY18MH010.aspx
- ISO/PAS 8800:2024; ISO/IEC TR 5469:2024; Regulation (EU) 2024/1689.

---
**Navigation:** ← [05 Conformal prediction](05_conformal_prediction.ipynb) · [Module index](README.md) · [07 End-to-end and foundation models](07_end_to_end_and_foundation_models.ipynb) →

### Further practice

**Try this:** change the baseline of integrated gradients (black image vs blurred input) and the patch size of the Shapley explanation, and compare the localisation scores; add a data-randomisation test (retrain on shuffled labels).

The author's companion repository [AI-Perception-Safety-Workshop](https://github.com/milinpatel07/AI-Perception-Safety-Workshop) contains an XAI exercise on real images.